# Chapter 01 Descriptive Statistics: rendered figures

Executed preview for browsing on GitHub (static image output, since GitHub's notebook viewer cannot run the interactive JS the live book uses). The source of truth for regenerating the book's figures is `chapter-01-descriptive-statistics-plots.py`. Run that script directly, not this notebook, to update `_generated/`.

In [1]:
"""
Interactive figures for Chapter 1: Introduction to Descriptive Statistics.

Every function builds one self-contained, standalone Plotly HTML page and writes it
to ../_generated/. The matching chapter-01-descriptive-statistics.md file embeds each
page in an <iframe>, so the chapter never depends on a live Python kernel to render.

Run directly to regenerate every figure:
    python chapters/chapter-01-descriptive-statistics-plots.py

The running example throughout is a simulated API latency dataset. It is a synthetic
log-normal distribution built to have the same shape (right-skewed, long tail) that
production latency logs almost always have, chosen specifically because "mean vs.
median" is a decision engineers make every time they set an alert threshold. It is
labeled as simulated everywhere it appears; it is not a claim about any company's
production traffic.
"""

import os

import numpy as np
import plotly.graph_objects as go
from scipy import stats

RNG = np.random.default_rng(42)
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "_generated")
os.makedirs(OUT_DIR, exist_ok=True)


In [2]:
def save(fig: go.Figure, name: str) -> str:
    path = os.path.join(OUT_DIR, f"{name}.html")
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    return path


In [3]:


def simulated_latency_ms(n: int = 5000, contamination: float = 0.0) -> np.ndarray:
    """Log-normal 'normal traffic' latency, optionally contaminated with a slow tail
    (e.g. a scheduled batch job or a cold-start burst hitting the same endpoint)."""
    base = RNG.lognormal(mean=np.log(45), sigma=0.35, size=n)
    n_slow = int(n * contamination)
    if n_slow:
        slow = RNG.lognormal(mean=np.log(600), sigma=0.5, size=n_slow)
        base = np.concatenate([base[: n - n_slow], slow])
    return base


In [4]:


# ---------------------------------------------------------------------------
# Figure 1: mean vs. median as the slow-request contamination fraction rises
# ---------------------------------------------------------------------------
def fig_mean_vs_median() -> go.Figure:
    fractions = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]
    frames = []
    for frac in fractions:
        data = simulated_latency_ms(contamination=frac)
        mean_v, median_v = float(np.mean(data)), float(np.median(data))
        hist = np.histogram(data, bins=np.linspace(0, 1200, 80))
        frames.append(
            go.Frame(
                name=f"{frac:.0%}",
                data=[go.Bar(x=hist[1][:-1], y=hist[0], marker_color="#4C78A8")],
                layout=go.Layout(
                    shapes=[
                        dict(type="line", x0=mean_v, x1=mean_v, y0=0, y1=1,
                             yref="paper", line=dict(color="#E45756", width=2, dash="solid")),
                        dict(type="line", x0=median_v, x1=median_v, y0=0, y1=1,
                             yref="paper", line=dict(color="#54A24B", width=2, dash="solid")),
                    ],
                    annotations=[
                        dict(x=mean_v, y=0.97, yref="paper", showarrow=False,
                             xanchor="left", text=f"mean = {mean_v:.0f} ms",
                             font=dict(color="#E45756")),
                        dict(x=median_v, y=0.88, yref="paper", showarrow=False,
                             xanchor="right", text=f"median = {median_v:.0f} ms",
                             font=dict(color="#54A24B")),
                    ],
                ),
            )
        )

    fig = go.Figure(data=frames[0].data, frames=frames, layout=frames[0].layout)
    fig.update_layout(
        title="Simulated API latency: mean vs. median as slow requests creep in",
        xaxis_title="Latency (ms)",
        yaxis_title="Requests",
        xaxis_range=[0, 1200],
        sliders=[{
            "active": 0,
            "currentvalue": {"prefix": "Slow-request share: "},
            "steps": [
                {"label": f.name, "method": "animate",
                 "args": [[f.name], {"mode": "immediate", "frame": {"duration": 300}}]}
                for f in frames
            ],
        }],
        margin=dict(t=70, l=60, r=30, b=50),
    )
    return fig
